# day-07-finetune-vs-rag-vs-prompt — worked solutions & answer key

Solutions to the exercises in [`../lesson.ipynb`](../lesson.ipynb), plus the self-check answer key. **Try each exercise yourself first** — the value is in the attempt, not the answer.

In [9]:
# ---- Solution 1 ----
def rag_k(q, topic, k):
    chunks, sims = retrieve(q, k)
    mem = [f"{c.split('::')[0].strip()} :: {c.split('::')[1].strip()}" for c in chunks]
    return base_model(STYLE_SYS, q, memory=mem)
print("top-3:", eval_approach(lambda q,t: rag_k(q,t,3), "RAG top-3"))

DOC_EMB_BROKEN = DOC_EMB.copy()
DOC_EMB_BROKEN[list(HANDBOOK).index("expenses")] = 0    # break one doc's retrievability
def retrieve_broken(query, k=1):
    sims = DOC_EMB_BROKEN @ embed(query); idx = np.argsort(-sims)[:k]
    return [DOCS[i] for i in idx]
def rag_broken(q, topic):
    chunks = retrieve_broken(q, 1)
    mem = [f"{c.split('::')[0].strip()} :: {c.split('::')[1].strip()}" for c in chunks]
    return base_model(STYLE_SYS, q, memory=mem)
print("with one doc unretrievable:", eval_approach(rag_broken, "RAG (broken embedding)"))

RAG top-3                          accuracy=1.00  mean_latency=1.23ms
top-3: 1.0


RAG (broken embedding)             accuracy=0.50  mean_latency=1.27ms
with one doc unretrievable: 0.5


In [10]:
# ---- Solution 2 ----
HANDBOOK_V2 = dict(HANDBOOK); HANDBOOK_V2["vacation"] = ("§4.1", "Full-time employees accrue 2.0 vacation days per month, max 30 days carryover.")
DOCS2 = [f"{k} :: {v[0]} {v[1]}" for k,v in HANDBOOK_V2.items()]
DOC_EMB2 = np.array([embed(d) for d in DOCS2])
def retrieve2(q,k=1):
    idx = np.argsort(-(DOC_EMB2 @ embed(q)))[:k]; return [DOCS2[i] for i in idx]
def rag2(q,t):
    c = retrieve2(q,1); mem=[f"{x.split('::')[0].strip()} :: {x.split('::')[1].strip()}" for x in c]
    return base_model(STYLE_SYS,q,memory=mem)
print("RAG after policy change (no retrain):", rag2("how many vacation days do I accrue per month?", "vacation")[0])
print("fine-tuned model (still stale)      :", ft(None, "how many vacation days do I accrue per month?")[0])

RAG after policy change (no retrain): Per policy §4.1: §4.1 Full-time employees accrue 2.0 vacation days per month, max 30 days carryover.
fine-tuned model (still stale)      : Per policy §4.1: Full-time employees accrue 1.75 vacation days per month, max 30 days carryover.


In [11]:
# ---- Solution 3 ----
months = 12
def cum(tok, calls): return FT_FIXED*(tok==FT_TOK) + calls*months*tok*PRICE
lo, hi = 1, 5_000_000
while hi-lo > 100:
    mid = (lo+hi)//2
    if cum(FT_TOK, mid) < cum(RAG_TOK, mid): hi = mid
    else: lo = mid
print(f"S3: fine-tune beats RAG over 12 months above ~{hi:,} calls/month")

S3: fine-tune beats RAG over 12 months above ~1,282,118 calls/month


In [12]:
# ---- Solution 4 ----
def hybrid_answer(q, topic):
    # style + schema discipline: baked in by fine-tuning (always "Per policy §X: ...")
    # fact: pulled fresh from retrieval (top-3, majority section), never frozen in weights
    chunks = retrieve(q, 3)[0]
    fact = chunks[0].split("::")[1].strip()
    sec = fact.split()[0]
    return f"Per policy {sec}: {' '.join(fact.split()[1:])}", 0.0015

print("hybrid overall:", eval_approach(hybrid_answer, "hybrid (FT style + RAG fact)"))
print("\nheld-out topics — where standalone fine-tuning failed:")
for q, t, frag in QUESTIONS:
    if t in {"security", "referral"}:
        h = hybrid_answer(q, t)[0]
        f = ft(None, q)[0]
        print(f"  {t:9s} hybrid: {'OK ' if frag.lower() in h.lower() else 'XX '}{h[:60]}")
        print(f"  {t:9s} FT   : {'OK ' if frag.lower() in f.lower() and f.startswith('Per policy') else 'XX '}{f[:60]}")

hybrid (FT style + RAG fact)       accuracy=0.50  mean_latency=1.50ms
hybrid overall: 0.5

held-out topics — where standalone fine-tuning failed:
  referral  hybrid: OK Per policy §5.6: Referral bonus is $2,000, paid after the re
  referral  FT   : XX Referral bonuses are commonly between $1,000 and $5,000.
  security  hybrid: OK Per policy §9.5: Laptops must be full-disk encrypted and loc
  security  FT   : XX Company laptops usually must be encrypted and password prote


### Solutions 5 & 6

**S5 — "add a new policy section":**
- *Prompt (full-handbook):* paste one more line into the system prompt. **~2 min**, but now
  every call is a bit bigger and you're closer to the window limit.
- *RAG:* add the doc, run the embedder on it, upsert to the vector store. **~15 min**, fully
  automatable, scales.
- *Fine-tune:* write training examples for the new section, re-run training, re-evaluate the
  whole model for regressions, redeploy the adapter. **1–3 days.**

**S6 — fine-tuned on the product catalog, prices now stale:** prices are *fast-changing facts*,
which is exactly what fine-tuning is worst at — they were frozen into the weights at training
time. Fix: pull prices out of the weights and into **RAG** (or a live tool/DB lookup at
inference). Keep fine-tuning, if anything, for *how* the model talks about products, not *what
the numbers are*.

### Answer key
1. Prompt = working memory (this call only); RAG = an open book on the desk (looked up per
   call); fine-tune = what was learned in school (in the weights, always on).
2. RAG. Prompt can't fit 900 pages in the window; fine-tuning would need a retrain every month
   and still risks stale/blended facts. RAG updates by re-embedding changed pages.
3. Fine-tune for the format (+ constrained decoding). Precondition: you have or can create
   enough labeled examples, and 5M calls/month easily amortises the fixed training cost;
   also verify prompt/RAG have genuinely plateaued first.
4. Retrieval accuracy — if the right chunk isn't retrieved, the generator is grounded in the
   wrong context and confidently wrong.
5. FAQ answers are facts that change; fine-tuning freezes them into weights at training time,
   so any change makes the model wrong until the next retrain. Facts belong in RAG.
6. e.g. a support assistant: **fine-tuned** for house voice, refusal rules, and tool-calling
   format; **RAG** over the current help center + the user's account data; a thin **system
   prompt** for the session context and safety reminders.